In [1]:
import os, sys, time, json, shutil, subprocess, warnings
warnings.filterwarnings("ignore")

try:
    import pyspark
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark"], check=True)
if shutil.which("java") is None:
    subprocess.run("apt-get -qq install -y openjdk-17-jdk-headless > /dev/null", shell=True, check=True)

from google.colab import drive
drive.mount("/content/drive")
PERSIST_DIR = "/content/drive/MyDrive/7006SCN_GSOD"
SCRATCH_DIR = "/content/gsod_scratch"
LOCAL_DIR   = "/content/gsod_local"

def sp(path):
    return path

RAW_DIR     = SCRATCH_DIR + "/raw_csv"
ARCHIVE_DIR = PERSIST_DIR + "/archives"
PARQUET_RAW = LOCAL_DIR + "/parquet/gsod_raw"
FEATURES    = LOCAL_DIR + "/parquet/gsod_features"
PRED_DIR    = LOCAL_DIR + "/parquet/predictions"
MODELS_DIR  = LOCAL_DIR + "/models"
OUT_DIR     = PERSIST_DIR + "/outputs"
for d in [PERSIST_DIR, SCRATCH_DIR, LOCAL_DIR, ARCHIVE_DIR, MODELS_DIR, OUT_DIR]:
    os.makedirs(d, exist_ok=True)

def backup_to_drive(local_path):
    dest = local_path.replace(LOCAL_DIR, PERSIST_DIR)
    shutil.rmtree(dest, ignore_errors=True)
    shutil.copytree(local_path, dest)
    print("backed up ->", dest)

def restore_from_drive(local_path):
    if not os.path.exists(local_path):
        shutil.copytree(local_path.replace(LOCAL_DIR, PERSIST_DIR), local_path)
        print("restored <-", local_path)

YEARS       = [2020, 2021, 2022, 2023]
TRAIN_YEARS = [2020, 2021, 2022]
TEST_YEAR   = 2023
SEED        = 42

from pyspark.sql import SparkSession, functions as F, Window
from pyspark import StorageLevel
TOTAL_RAM_GB = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
DRIVER_MEM = f"{max(2, int(TOTAL_RAM_GB * 0.7))}g"

spark = (SparkSession.builder
         .master("local[*]")
         .appName("7006SCN_GSOD_Task6")
         .config("spark.driver.memory", DRIVER_MEM)
         .config("spark.sql.shuffle.partitions", "64")
         .config("spark.sql.adaptive.enabled", "true")
         .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
         .config("spark.sql.files.maxPartitionBytes", str(128 * 1024 * 1024))
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

def n_partitions(df):
    return df.rdd.getNumPartitions()

def safe_persist(df, name, level=StorageLevel.MEMORY_AND_DISK):
    out = df.persist(level)
    out.count()
    return out

def save_csv(pdf, name):
    path = f"{OUT_DIR}/{name}"
    pdf.to_csv(path, index=False)
    print("saved ->", path)

print(f"Spark {spark.version} | master={spark.sparkContext.master} | cores={spark.sparkContext.defaultParallelism} "
      f"| driver memory={DRIVER_MEM} | RAM={TOTAL_RAM_GB:.1f} GB")
print("PERSIST_DIR =", PERSIST_DIR)

restore_from_drive(FEATURES)
restore_from_drive(f"{PRED_DIR}/LogisticRegression")
restore_from_drive(f"{PRED_DIR}/DecisionTree")
restore_from_drive(f"{PRED_DIR}/RandomForest")
restore_from_drive(f"{PRED_DIR}/GBT")

Mounted at /content/drive
Spark 4.0.4 | master=local[*] | cores=2 | driver memory=8g | RAM=12.7 GB
PERSIST_DIR = /content/drive/MyDrive/7006SCN_GSOD
restored <- /content/gsod_local/parquet/gsod_features
restored <- /content/gsod_local/parquet/predictions/LogisticRegression
restored <- /content/gsod_local/parquet/predictions/DecisionTree
restored <- /content/gsod_local/parquet/predictions/RandomForest
restored <- /content/gsod_local/parquet/predictions/GBT


In [2]:
import pandas as pd, numpy as np, zipfile, glob
TAB_DIR = f"{PERSIST_DIR}/tableau"
os.makedirs(TAB_DIR, exist_ok=True)
EXPORTS = []

def export(pdf, name, round_to=4):
    pdf = pdf.copy()
    for c in pdf.select_dtypes("float").columns:
        pdf[c] = pdf[c].round(round_to)
    path = f"{TAB_DIR}/{name}"
    pdf.to_csv(path, index=False, encoding="utf-8")
    EXPORTS.append({"file": name, "rows": len(pdf), "columns": len(pdf.columns)})
    print(f"{name:<32} {len(pdf):>7,} rows")

def out(name):
    return pd.read_csv(f"{OUT_DIR}/{name}")

features = spark.read.parquet(sp(FEATURES))
best_model = pd.read_csv(f"{OUT_DIR}/task5_metrics.csv").sort_values("auc_pr", ascending=False).iloc[0].model
print("Best model (PR-AUC):", best_model)

Best model (PR-AUC): RandomForest


In [3]:
m = out("task2_missing_values.csv")
dq = m.melt(id_vars="column", value_vars=["missing_before", "missing_after_cleaning"], var_name="stage", value_name="missing_rows")
dq["stage"] = dq.stage.map({"missing_before": "Before (raw incl. sentinels)", "missing_after_cleaning": "After cleaning"})
dq["missing_rows"] = dq.missing_rows.fillna(0)
dq["note"] = np.where(dq.column == "STP", "column dropped", "")
export(dq, "dq_missing.csv")
export(out("task2_pipeline_funnel.csv"), "dq_pipeline_funnel.csv")
export(out("task2_class_balance.csv"), "dq_class_balance.csv")
export(out("task1_rows_per_year.csv"), "dq_rows_per_year.csv")
export(out("task4_partition_sizes.csv"), "dq_partitions.csv")

dq_missing.csv                        24 rows
dq_pipeline_funnel.csv                 5 rows
dq_class_balance.csv                   4 rows
dq_rows_per_year.csv                   4 rows
dq_partitions.csv                     16 rows


In [4]:
met = out("task5_metrics.csv")
ref = out("task3_model_summary.csv")
ref = ref[ref.model == "Persistence (naive)"]
met_all = pd.concat([met, ref], ignore_index=True)
long = met_all.melt(id_vars="model", value_vars=["accuracy", "precision", "recall", "f1", "balanced_accuracy", "auc_roc", "auc_pr"],
                    var_name="metric", value_name="value").dropna()
long["metric"] = long.metric.map({"accuracy": "Accuracy", "precision": "Precision", "recall": "Recall", "f1": "F1",
                                  "balanced_accuracy": "Balanced accuracy", "auc_roc": "ROC-AUC", "auc_pr": "PR-AUC"})
export(long, "mp_metrics_long.csv")

cur = out("task5_curve_points.csv")
cur = cur.groupby("model", group_keys=False).apply(lambda g: g.iloc[::5])
export(cur, "mp_curves.csv")
export(out("task5_confusion_long.csv"), "mp_confusion.csv")

fi = out("task3_feature_importance.csv")
shap_imp = out("task5_shap_importance.csv").rename(columns={"mean_abs_shap": "importance"}).assign(method="Mean |SHAP|")
fi = pd.concat([fi, shap_imp], ignore_index=True)
fi["rank"] = fi.groupby(["model", "method"]).importance.rank(ascending=False, method="first").astype(int)
export(fi, "mp_feature_importance.csv", round_to=6)

stab = out("task5_perturbation.csv")
stab["noise_pct"] = (stab.noise_k * 100).round(0)
export(stab, "mp_stability.csv")
export(out("task5_training_stability.csv"), "mp_training_stability.csv")

mp_metrics_long.csv                   32 rows
mp_curves.csv                        639 rows
mp_confusion.csv                      16 rows
mp_feature_importance.csv            264 rows
mp_stability.csv                      16 rows
mp_training_stability.csv              4 rows


In [5]:
pred = spark.read.parquet(sp(f"{PRED_DIR}/{best_model}"))
station = (pred.groupBy("STATION", "NAME", "CLIMATE_ZONE", "HEMISPHERE")
           .agg(F.round(F.avg("LATITUDE"), 3).alias("lat"), F.round(F.avg("LONGITUDE"), 3).alias("lon"),
                F.count("*").alias("days"), F.sum("label").alias("extreme_days"),
                F.avg("label").alias("observed_extreme_rate"), F.avg("prob").alias("mean_predicted_risk"),
                F.sum(F.when((F.col("label") == 1) & (F.col("prediction") == 1), 1).otherwise(0)).alias("caught"),
                F.sum(F.when((F.col("label") == 0) & (F.col("prediction") == 1), 1).otherwise(0)).alias("false_alarms"))
           .filter("days >= 30")
           .withColumn("recall", F.when(F.col("extreme_days") > 0, F.col("caught") / F.col("extreme_days")))
           .toPandas())
station["country_code"] = station.NAME.str.extract(r"([A-Z]{2})\s*$")[0]
station["risk_band"] = pd.cut(station.observed_extreme_rate, [-0.01, 0.05, 0.15, 0.30, 1.0],
                              labels=["Low (<5%)", "Moderate (5-15%)", "High (15-30%)", "Very high (>30%)"])
export(station, "bi_station_risk.csv")

monthly = (features.groupBy("YEAR", "MONTH", "CLIMATE_ZONE", "HEMISPHERE")
           .agg(F.count("*").alias("station_days"), F.avg("IS_EXTREME").alias("extreme_rate"),
                F.avg("TEMP").alias("avg_temp_F"), F.avg("PRCP").alias("avg_prcp_in"))
           .orderBy("YEAR", "MONTH").toPandas())
monthly["date"] = pd.to_datetime(dict(year=monthly.YEAR, month=monthly.MONTH, day=1)).dt.date
pm = (pred.groupBy("MONTH", "CLIMATE_ZONE", "HEMISPHERE")
      .agg(F.avg("prob").alias("predicted_risk"), F.avg("label").alias("observed_next_day_rate")).toPandas())
monthly = monthly.merge(pm.assign(YEAR=TEST_YEAR), on=["YEAR", "MONTH", "CLIMATE_ZONE", "HEMISPHERE"], how="left")
export(monthly, "bi_monthly_zone.csv")

types = (features.groupBy("SEASON", "CLIMATE_ZONE", "HEMISPHERE")
         .agg(*[F.sum(c).alias(c) for c in ["EXT_HEAT", "EXT_COLD", "EXT_RAIN", "EXT_WIND", "EXT_STORM"]],
              F.count("*").alias("station_days")).toPandas())
types = types.melt(id_vars=["SEASON", "CLIMATE_ZONE", "HEMISPHERE", "station_days"], var_name="extreme_type", value_name="days")
types["extreme_type"] = types.extreme_type.map({"EXT_HEAT": "Heat (MAX>=95F)", "EXT_COLD": "Cold (MIN<=-4F)",
                                                "EXT_RAIN": "Heavy rain (>=1in)", "EXT_WIND": "Gale wind",
                                                "EXT_STORM": "Storm (thunder/hail/tornado)"})
types["rate"] = types.days / types.station_days
export(types, "bi_extreme_types.csv")
export(out("task5_fairness.csv"), "bi_fairness.csv")
export(out("task5_thresholds.csv"), "bi_thresholds.csv")

bi_station_risk.csv               11,912 rows
bi_monthly_zone.csv                  384 rows
bi_extreme_types.csv                 160 rows
bi_fairness.csv                        6 rows
bi_thresholds.csv                      4 rows


In [6]:
export(out("task4_scalability.csv"), "sc_scalability.csv")
export(out("task4_shuffle_partitions.csv"), "sc_shuffle.csv")
export(out("task4_caching.csv"), "sc_caching.csv")
export(out("task4_repartitioning.csv"), "sc_repartitioning.csv")
export(out("task4_imbalance.csv"), "sc_imbalance.csv")
export(out("task4_resource_config.csv").astype(str), "sc_config.csv")
if os.path.exists(f"{OUT_DIR}/task4_spark_stages.csv"):
    st = out("task4_spark_stages.csv").sort_values("duration_s", ascending=False).head(25)
    st["name"] = st.name.str.slice(0, 60)
    export(st, "sc_stages.csv")
price = float(os.environ.get("GSOD_PRICE_PER_HOUR", 0.20))
s3 = out("task3_model_summary.csv")
s3 = s3[s3.model != "Persistence (naive)"].copy()
s3["cv_time_s"] = s3.cv_time_s.fillna(0)
s3["train_cost_GBP"] = s3.train_time_s / 3600 * price
s3["tuning_cost_GBP"] = s3.cv_time_s / 3600 * price
s3 = s3.merge(met[["model", "auc_pr"]].rename(columns={"auc_pr": "test_pr_auc"}), on="model")
export(s3[["model", "hyperparameters", "train_time_s", "cv_time_s", "train_cost_GBP", "tuning_cost_GBP", "test_pr_auc"]],
       "sc_model_cost.csv", round_to=6)

sc_scalability.csv                    12 rows
sc_shuffle.csv                         5 rows
sc_caching.csv                         3 rows
sc_repartitioning.csv                  4 rows
sc_imbalance.csv                       3 rows
sc_config.csv                         13 rows
sc_stages.csv                         25 rows
sc_model_cost.csv                      4 rows


In [7]:
pd.DataFrame(EXPORTS).to_csv(f"{TAB_DIR}/_manifest.csv", index=False)
zip_path = f"{PERSIST_DIR}/tableau_extracts.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"{TAB_DIR}/*.csv"):
        z.write(f, os.path.basename(f))
print(pd.DataFrame(EXPORTS).to_string(index=False))
print(f"\nAll extracts zipped -> {zip_path}  ({os.path.getsize(zip_path)/1e6:.2f} MB)")
from google.colab import files
files.download(zip_path)

                     file  rows  columns
           dq_missing.csv    24        4
   dq_pipeline_funnel.csv     5        3
     dq_class_balance.csv     4        4
     dq_rows_per_year.csv     4        3
        dq_partitions.csv    16        2
      mp_metrics_long.csv    32        3
            mp_curves.csv   639        6
         mp_confusion.csv    16        4
mp_feature_importance.csv   264        5
         mp_stability.csv    16        8
mp_training_stability.csv     4        6
      bi_station_risk.csv 11912       15
      bi_monthly_zone.csv   384       11
     bi_extreme_types.csv   160        7
          bi_fairness.csv     6        8
        bi_thresholds.csv     4        5
       sc_scalability.csv    12        7
           sc_shuffle.csv     5        2
           sc_caching.csv     3        3
    sc_repartitioning.csv     4        3
         sc_imbalance.csv     3        6
            sc_config.csv    13        2
            sc_stages.csv    25        9
        sc_model

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>